In [1]:
!unzip STANDUP_RGB_P.zip

Archive:  STANDUP_RGB_P.zip
   creating: content/STANDUP_RGB_P/
  inflating: content/STANDUP_RGB_P/H_0098.png  
  inflating: content/STANDUP_RGB_P/R0_0098.png  
  inflating: content/STANDUP_RGB_P/R0_0099.png  
  inflating: content/STANDUP_RGB_P/H_0105.png  
  inflating: content/STANDUP_RGB_P/H_0009.png  
  inflating: content/STANDUP_RGB_P/R0_0012.png  
  inflating: content/STANDUP_RGB_P/R0_0121.png  
  inflating: content/STANDUP_RGB_P/R1_0068.png  
  inflating: content/STANDUP_RGB_P/R1_0037.png  
  inflating: content/STANDUP_RGB_P/R1_0048.png  
  inflating: content/STANDUP_RGB_P/R0_0016.png  
  inflating: content/STANDUP_RGB_P/R1_0026.png  
  inflating: content/STANDUP_RGB_P/R1_0015.png  
  inflating: content/STANDUP_RGB_P/R2_0006.png  
  inflating: content/STANDUP_RGB_P/R0_0107.png  
  inflating: content/STANDUP_RGB_P/R1_0014.png  
  inflating: content/STANDUP_RGB_P/R2_0021.png  
  inflating: content/STANDUP_RGB_P/R2_0015.png  
  inflating: content/STANDUP_RGB_P/H_0078.png  
  inflati

In [2]:
import os, numpy as np, pandas as pd, torch, torch.nn as nn, joblib
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import EfficientNet_B3_Weights
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
ORDINAL_MAP = {"healthy": 0, "R0": 1, "R1": 2, "R2": 3}
NUM_CLASSES = 4

In [3]:
rgb_meta = pd.read_csv("/content/standup_rgb_metadata.csv")
thermal_feat = pd.read_csv("/content/standup_thermal_features.csv")
rgb_meta["ordinal_label"] = rgb_meta["label"].map(ORDINAL_MAP)
thermal_feat["ordinal_label"] = thermal_feat["label"].map(ORDINAL_MAP)

patients = rgb_meta[["patient_id", "label"]].drop_duplicates()
train_pid, test_pid = train_test_split(patients["patient_id"], test_size=0.2, random_state=SEED, stratify=patients["label"])
rgb_meta["split"] = rgb_meta["patient_id"].apply(lambda p: "train" if p in set(train_pid) else "test")
thermal_feat["split"] = thermal_feat["patient_id"].apply(lambda p: "train" if p in set(train_pid) else "test")

# real pairing: same filename = same patient/timepoint across modalities
paired = rgb_meta.merge(thermal_feat, on=["filename","patient_id","timepoint","label","ordinal_label","split"], suffixes=("_rgb","_thermal"))
print(paired.shape)
print(paired["split"].value_counts())

(415, 17)
split
train    331
test      84
Name: count, dtype: int64


In [4]:
class CoralLayer(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.fc = nn.Linear(in_features, 1, bias=False)
        self.bias = nn.Parameter(torch.zeros(num_classes-1))
    def forward(self, x):
        return self.fc(x) + self.bias

weights = EfficientNet_B3_Weights.IMAGENET1K_V1
rgb_model = models.efficientnet_b3(weights=weights)
in_features = rgb_model.classifier[1].in_features
rgb_model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_features,256), nn.ReLU(), nn.Dropout(0.3))
rgb_model.coral = CoralLayer(256, NUM_CLASSES)
rgb_model.load_state_dict(torch.load("/content/best_rgb_severity_coral.pth", map_location=DEVICE))
rgb_model = rgb_model.to(DEVICE).eval()

thermal_classifiers = joblib.load("/content/best_thermal_severity_ordinal.joblib")
print("Models loaded.")

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 79.8MB/s]


Models loaded.


In [6]:
IMG_SIZE = 224
IMAGENET_MEAN, IMAGENET_STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
infer_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def rgb_threshold_probs(paths, batch_size=32):
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(paths), batch_size):
            imgs = torch.stack([infer_tf(Image.open(p).convert("RGB")) for p in paths[i:i+batch_size]]).to(DEVICE)
            feat = rgb_model.classifier(rgb_model.features(imgs).mean([2,3]))
            logits = rgb_model.coral(feat)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

paired["rgb_full_path"] = paired["patient_id"].apply(lambda x: None)  # placeholder, filled below
paired["rgb_full_path"] = paired["filename"].apply(lambda f: os.path.join("/content/STANDUP_RGB_P", f))
rgb_probs = rgb_threshold_probs(paired["rgb_full_path"].tolist())
paired["p_rgb_0"], paired["p_rgb_1"], paired["p_rgb_2"] = rgb_probs[:,0], rgb_probs[:,1], rgb_probs[:,2]
print(rgb_probs.shape)

(415, 3)


In [7]:
THERMAL_FEATURE_COLS = ["mean_temp","max_temp","std_temp","range_temp","hot_spot_count",
                         "temp_gradient","glcm_contrast","glcm_homogeneity","glcm_entropy"]

X_all = paired[THERMAL_FEATURE_COLS].values
thermal_probs = np.stack([c.predict_proba(X_all)[:,1] for c in thermal_classifiers], axis=1)
paired["p_thermal_0"], paired["p_thermal_1"], paired["p_thermal_2"] = thermal_probs[:,0], thermal_probs[:,1], thermal_probs[:,2]
print(thermal_probs.shape)

(415, 3)


In [8]:
for k in range(3):
    paired[f"p_fused_{k}"] = 0.5*paired[f"p_rgb_{k}"] + 0.5*paired[f"p_thermal_{k}"]

def ordinal_from_probs(df, cols):
    probs = df[cols].values
    return (probs > 0.5).sum(axis=1)

paired["pred_rgb"] = ordinal_from_probs(paired, ["p_rgb_0","p_rgb_1","p_rgb_2"])
paired["pred_thermal"] = ordinal_from_probs(paired, ["p_thermal_0","p_thermal_1","p_thermal_2"])
paired["pred_fused"] = ordinal_from_probs(paired, ["p_fused_0","p_fused_1","p_fused_2"])

test_df = paired[paired.split=="test"]
for name, col in [("RGB only","pred_rgb"), ("Thermal only","pred_thermal"), ("Fused (equal weight)","pred_fused")]:
    acc = accuracy_score(test_df["ordinal_label"], test_df[col])
    qwk = cohen_kappa_score(test_df["ordinal_label"], test_df[col], weights="quadratic")
    print(f"{name}: Acc={acc:.4f} QWK={qwk:.4f}")

paired.to_csv("/content/standup_fusion_probs.csv", index=False)

RGB only: Acc=0.5000 QWK=0.5694
Thermal only: Acc=0.3810 QWK=0.4139
Fused (equal weight): Acc=0.5119 QWK=0.5814
